# Hybrid Search (BM25 + Dense for retrieval & RRF for top k)

<img src="../images/BM25-Dense-RRF.png" width=“500”>

In [ ]:
from langchain_core.globals import set_debug
set_debug(False)

## Initialize Azure Chat OpenAI

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv(override=True, verbose=True)

chat_llm = AzureChatOpenAI(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
)

## Initialize Azure Embedding Open AI

In [ ]:
emd_llm = AzureOpenAIEmbeddings(
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_deployment=os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
)

## Load the Document and upload it to vector store

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

file = '../Data/HTTP-Status-Codes.pdf'
pdf_docs = PyMuPDFLoader(file_path=file).load()


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)
chunks = splitter.split_documents(documents=pdf_docs)

In [ ]:
from langchain_community.vectorstores import FAISS

faiss = FAISS.from_documents(documents=chunks, embedding=emd_llm)


## User question 

In [ ]:
num_queries = 3
user_question = "What is error code 404?"

## Call BM25 store (e.g. Elastic Search, Qdrant etc) 

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(pdf_docs)
bm25_chunks = bm25_retriever.invoke(user_question)

In [ ]:
print(f"BM25 chunk size: {len(bm25_chunks)}")
bm25_top_chunks = []
for c in bm25_chunks:
    print(c.page_content, end="\n-----\n")
    bm25_top_chunks.append(c.page_content)


## Dense retrieval

In [ ]:

faiss_chunks = faiss.similarity_search_with_score(query=user_question, k=num_queries)

In [ ]:
print(f"Chunks fetched from FAISS Vector store: {len(faiss_chunks)}")
faiss_top_chunks = []
for c, score in faiss_chunks:
    print(score)
    print(c.page_content, end="\n-----\n")
    
    faiss_top_chunks.append(c.page_content)

## Fuse the docs using RRF 

In [34]:
from collections import defaultdict

k = 60
rrf_scores = defaultdict(float)
ranked_lists = [bm25_top_chunks, faiss_top_chunks]

# Iterate through each retrieval system's results
for rank_list in ranked_lists:
    for rank, item in enumerate(rank_list):
        # rank starts at 0 in Python, so we add 1 to get 1-based rank position
        rrf_scores[item] += 1.0 / (k + (rank + 1))
        
# Sort items based on scores in descending order
sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)



In [37]:
fetched_response = ""
for c, score in sorted_results:
    print(f"{c}\n{score}", end="\n-----\n")
    fetched_response += c

How to Transfer Your Contabo Vps to Another Region (Video)
What is the Customer Control Panel and How do I Access it?
Can You Provide More Information On Contabo’s Horizen Node Image?
What Limits are There on Object Storage?
How Do I Access Full Monitoring?
Products
Solutions
Locations
Company
Contact & Help
Keep in touch
Latest News and Deals from us. No spam guarantee!
Email
Enter your email address
Subscribe
The  privacy statement  holds information on the involved data processing and right to withdraw consent.
Follow Us
19/05/2026, 08:52
HTTP Response Codes and Server Statuses - Complete Reference | Contabo Blog
https://contabo.com/blog/http-response-codes-server-statuses/?utm_source=google&utm_medium=cpc&utm_campaign=core-prospecting-india-pmax&utm_te…
20/21
0.01639344262295082
-----
permanent.
4XX codes flag client errors. You typed a wrong URL. You don’t have
permission. The page doesn’t exist. These are on you or your users.
0.01639344262295082
-----
2XX Success Codes Complete 

In [38]:
from langchain_core.prompts import ChatPromptTemplate
PROMPT_TEMPLATE = """"
You are a helpful assistance. You answer user's query {question} from the provided context {context}. If context isn't sufficient to provide the answer, inform the user that Context isn't sufficient to provide the answer of your query.
"""
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT_TEMPLATE),
        ("user",  "user query: {question}"),
    ]
)


In [39]:
prompt =  prompt_template.invoke({'question': user_question, 'context': fetched_response})
response = chat_llm.invoke(prompt)
print(response)

print(response.content)

content='A 404 is the HTTP "Not Found" response code. It means the server can’t find the resource you requested — typically because the URL is wrong, the page was removed, or the resource is temporarily unavailable. It’s a client-side (4xx) error.\n\nWhy it matters\n- Users see a broken page and often leave, hurting UX.\n- Search engines may drop 404 pages from their index, which can reduce traffic and lose link equity.\n\nCommon causes\n- Typo or bad link\n- Page deleted or moved\n- Incorrect routing or missing permissions/configuration\n\nQuick fixes and best practices\n- If the page should exist, restore it or fix the URL/link.\n- If the content moved, use a 301 redirect to the new location.\n- If the page was intentionally removed, consider returning 410 Gone to tell search engines it’s permanent.\n- Create a helpful custom 404 page with navigation and a search box to keep visitors on your site.\n- Monitor and find 404s with Google Search Console, server logs, or crawlers like Scre